# Model Comparison

This notebook compares all four classifiers trained on the preprocessed heart disease data:
- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbours (KNN)

Comparison includes:
- Metrics table (Accuracy, Precision, Recall, F1, AUC-ROC)
- Bar chart comparison
- ROC curves overlay
- Cross-validation box plots

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)

# Load preprocessed train/test data
train_df = pd.read_csv("heart_train_preprocessed.csv")
test_df = pd.read_csv("heart_test_preprocessed.csv")

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 1. Train All Models

Each model uses the same hyperparameters and tuning strategy as its individual notebook.

In [ ]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print("Logistic Regression trained.")

# --- Decision Tree (with GridSearchCV) ---
dt_param_grid = {
    "max_depth": [3, 4, 5, 6, 7, 8, 10, 12, 15, None],
    "min_samples_split": [2, 3, 5, 7, 10, 15],
    "min_samples_leaf": [1, 2, 3, 5, 7],
    "criterion": ["gini", "entropy"],
    "ccp_alpha": [0.0, 0.001, 0.005, 0.01, 0.02, 0.05],
}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_param_grid, cv=10, scoring="f1", n_jobs=-1)
dt_grid.fit(X_train, y_train)
dt_model = dt_grid.best_estimator_
print(f"Decision Tree trained. Best params: {dt_grid.best_params_}")

# --- Random Forest (with GridSearchCV) ---
rf_param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 8, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"],
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring="f1", n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_model = rf_grid.best_estimator_
print(f"Random Forest trained. Best params: {rf_grid.best_params_}")

# --- KNN (with GridSearchCV) ---
knn_param_grid = {
    "n_neighbors": list(range(1, 31)),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=5, scoring="f1", n_jobs=-1)
knn_grid.fit(X_train, y_train)
knn_model = knn_grid.best_estimator_
print(f"KNN trained. Best params: {knn_grid.best_params_}")

## 2. Metrics Comparison Table

In [ ]:
models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
    "KNN": knn_model,
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

## 3. Metrics Bar Chart

In [ ]:
results_df.plot(kind="bar", figsize=(12, 6), rot=0, edgecolor="black", linewidth=0.5)
plt.title("Model Comparison — Test Set Metrics")
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 4. ROC Curves Overlay

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All Models")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Cross-Validation Comparison (Box Plot)

In [ ]:
# 10-fold CV for each model
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=10, scoring="f1")
    cv_results[name] = scores
    print(f"{name:25s}  Mean F1: {scores.mean():.4f}  Std: {scores.std():.4f}")

# Box plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor="#1f77b4", alpha=0.6),
           medianprops=dict(color="red", linewidth=2))
ax.set_ylabel("F1-score")
ax.set_title("10-Fold Cross-Validation — F1-score Distribution")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
# Find best model by each metric
print("Best model by each metric (test set):")
for col in results_df.columns:
    best = results_df[col].idxmax()
    print(f"  {col:12s}: {best} ({results_df.loc[best, col]:.4f})")

print(f"\nBest model by mean 10-Fold CV F1:")
best_cv = max(cv_results, key=lambda k: cv_results[k].mean())
print(f"  {best_cv} ({cv_results[best_cv].mean():.4f})")